# 数据统计  
在神经网络的计算过程中，经常需要统计数据的各种属性，如最值、最值位置、均值、范数等信息。由于张量通常较大，直接观察数据很难获得有用信息，通过获取这些张
量的统计信息可以较轻松地推测张量数值的分布

## 1.向量范数 tf.norm(x, ord)  
向量范数(Vector Norm)是表征向量“长度”的一种度量方法，它可以推广到张量上。
在神经网络中，常用来表示张量的权值大小，梯度大小等.
- L1 范数，定义为向量𝒙的所有元素绝对值之和
- L2 范数，定义为向量𝒙的所有元素的平方和，再开根号
- ∞ −范数，定义为向量𝒙的所有元素绝对值的最大值

在 TensorFlow 中，可以通过 tf.norm(x, ord)求解张量的 L1、L2、∞等范数，其中参数
ord 指定为 1、2 时计算 L1、L2 范数，指定为 np.inf 时计算∞ −范数

In [3]:
import tensorflow as tf
import numpy as np

x = tf.ones([2, 2])
tf.norm(x, ord=1)  # 计算L1范数

<tf.Tensor: shape=(), dtype=float32, numpy=4.0>

In [2]:
tf.norm(x, ord=2)  # 计算 L2 范数

<tf.Tensor: shape=(), dtype=float32, numpy=2.0>

In [4]:
tf.norm(x, ord=np.inf)  # 计算∞范数

<tf.Tensor: shape=(), dtype=float32, numpy=1.0>

## 2.最值、均值、和 tf.reduce_max、tf.reduce_min、tf.reduce_mean、tf.reduce_sum  
通过 tf.reduce_max、tf.reduce_min、tf.reduce_mean、tf.reduce_sum 函数可以求解张量
在某个维度上的最大、最小、均值、和，也可以求全局最大、最小、均值、和信息。

考虑 shape 为[4,10]的张量，其中，第一个维度代表样本数量，第二个维度代表了当前
样本分别属于 10 个类别的概率，需要求出每个样本的概率最大值为，可以通过
tf.reduce_max 函数实现

In [ ]:
x = tf.random.normal([4, 10])
tf.reduce_max(x, axis=1)  # 统计概率维度上的最大值
# 返回长度为 4 的向量，分别代表了每个样本的最大概率值。

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([0.8728627, 1.6736233, 0.9086078, 1.8624841], dtype=float32)>

In [7]:
# 同样求出每个样本概率的最小值，实现如下：
tf.reduce_min(x, axis=1)

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([-1.6181647, -0.8585464, -2.6381876, -2.581572 ], dtype=float32)>

In [8]:
# 求出每个样本的概率的均值
tf.reduce_mean(x, axis=1)

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([-0.16528599,  0.14927928, -0.36555833, -0.24177957], dtype=float32)>

### 当不指定 axis 参数时，tf.reduce_*函数会求解出全局元素的最大、最小、均值、和等数据

In [9]:
x = tf.random.normal([4, 10])
tf.reduce_max(x), tf.reduce_min(x), tf.reduce_mean(x), tf.reduce_sum(x)

(<tf.Tensor: shape=(), dtype=float32, numpy=2.8390229>,
 <tf.Tensor: shape=(), dtype=float32, numpy=-2.1135852>,
 <tf.Tensor: shape=(), dtype=float32, numpy=0.0068798126>,
 <tf.Tensor: shape=(), dtype=float32, numpy=0.2751925>)

在求解误差函数时，通过 TensorFlow 的 MSE 误差函数可以求得每个样本的误差，需
要计算样本的平均误差，此时可以通过 tf.reduce_mean 在样本数维度上计算均值，实现如
下

In [ ]:
from tensorflow import keras

out = tf.random.normal([4, 10])  # 模拟网络预测输出
y = tf.constant([1, 2, 2, 0])
y = tf.one_hot(y, depth=10)
loss = keras.losses.mse(y, out)  # 计算每个样本的误差
loss = tf.reduce_mean(loss)  # 平均误差，在样本数维度上取均值
loss  # 误差标量

<tf.Tensor: shape=(), dtype=float32, numpy=0.906191>

与均值函数相似的是求和函数 tf.reduce_sum(x, axis)，它可以求解张量在 axis 轴上所有
特征的和

In [13]:
out = tf.random.normal([4, 10])
tf.reduce_sum(out, axis=-1)

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 3.0588174, -3.5267248,  1.1739199, -3.5453405], dtype=float32)>

### tf.argmax(x, axis)和 tf.argmin(x, axis)可以求解在 axis 轴上，x 的最大值、最小值所在的索引号  
除了希望获取张量的最值信息，还希望获得最值所在的位置索引号，例如分类任务的
标签预测，就需要知道概率最大值所在的位置索引号，一般把这个位置索引号作为预测类
别。考虑 10 分类问题，我们得到神经网络的输出张量 out，shape 为[2,10]，代表了 2 个样
本属于 10 个类别的概率，由于元素的位置索引代表了当前样本属于此类别的概率，预测时
往往会选择概率值最大的元素所在的索引号作为样本类别的预测值，  
以第一个样本为例，可以看到，它概率最大的索引为𝑖 = 0，最大概率值为 0.1877。由于每
个索引号上的概率值代表了样本属于此索引号的类别的概率，因此第一个样本属于 0 类的
概率最大，在预测时考虑第一个样本应该最有可能属于类别 0。这就是需要求解最大值的
索引号的一个典型应用

In [15]:
out = tf.random.normal([2, 10])
print('out_raw:', out)
out = tf.nn.softmax(out, axis=1)  # 通过 softmax 函数转换为概率值
out

out_raw: tf.Tensor(
[[-1.9223962   0.932173    0.7662972   0.718868   -0.91073596  1.0512422
  -0.07882502  1.5169137  -0.50560254 -0.7484147 ]
 [-0.4053648   0.27089262  0.93885875 -0.56604457  0.9124951  -1.3762738
  -1.1971635  -1.8124563   1.4151034  -0.903947  ]], shape=(2, 10), dtype=float32)


<tf.Tensor: shape=(2, 10), dtype=float32, numpy=
array([[0.00875146, 0.15198615, 0.12875529, 0.12279108, 0.02406794,
        0.17120448, 0.05530102, 0.27274302, 0.03608993, 0.02830962],
       [0.05195491, 0.10216954, 0.19925787, 0.04424297, 0.19407335,
        0.01967733, 0.02353709, 0.01272139, 0.32080853, 0.03155696]],
      dtype=float32)>

In [17]:
pred = tf.argmax(out, axis=1)
pred

<tf.Tensor: shape=(2,), dtype=int64, numpy=array([7, 8])>